# Recover original seq20 provenance
Read-only, bounded source investigation. Reuse Phase 2B evidence instead of reloading tensors, catalog records or rerunning overlap analysis. Search named catalog-related sources and top-level BiLSTM notebooks; inspect only relevant local history/checkpoints if present. Do not recurse through the project, runtime packages or waveform-generation directories. A load call, downstream 36-channel model, or save of model weights is not the original dataset constructor.

Archive inspection uses ZIP metadata and pickle **disassembly**, never pickle execution. No model fitting, data generation or dataset writes are included.

In [1]:
from pathlib import Path
import json,re,ast,hashlib,zipfile,pickletools,sqlite3,datetime
from urllib.parse import urlparse
ROOT=Path.cwd().resolve()
if (ROOT/"bilstm/bilstm_dataset").is_dir():ROOT=ROOT/"bilstm"
assert (ROOT/"bilstm_dataset").is_dir()
PROJECT=ROOT.parent
MEM=ROOT/".agent-memory"
prior=json.loads((ROOT/"phase2b_generalization_results/provenance_evidence.json").read_text())
def sha(p):
    h=hashlib.sha256()
    with Path(p).open("rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):h.update(chunk)
    return h.hexdigest()
datasets=[ROOT/"bilstm_dataset"/n for n in (
    "bilstm_dataset_seq20.pt","bilstm_dataset_seq20.npz","bilstm_dataset_seq20_station_grouped.npz")]
before={p.name:sha(p) for p in datasets}
assert before["bilstm_dataset_seq20.pt"]==prior["source_hashes"]["bilstm/bilstm_dataset/bilstm_dataset_seq20.pt"]
# Explicit allowlist; no recursive scan.
names=[
    "preprocess.py","preprocess_dl_temporal.py","preprocessing_report.txt","status.md",
    "bilstm/phase2_bilstm.ipynb","bilstm/phase2_bilstm_generalization_audit.ipynb",
    "bilstm/phase2b_generalization_experiment.ipynb",
    "bilstm/phase2_bilstm_results/experiment_manifest.json",
    "cnn/cnn wave/phase1_cnn_forecasting.ipynb",
    "hybrid/cnn_bilstm_attention_pytorch.py",
    "cnn/cnn wave/.agent-memory/dataset_index.md",
    "cnn/cnn wave/.agent-memory/project_context.md",
    "cnn/cnn wave/.agent-memory/experiment_registry.json",
    "cnn/cnn wave/data wave/README.md",
    "cnn/cnn wave/dataset_comparison_report.md",
    "cnn/cnn wave/phase1_cnn_results/reports/dataset_inventory.json",
]
# Only the known notebook checkpoint directories; missing directories are recorded.
checkpoint_dirs=[PROJECT/".ipynb_checkpoints",ROOT/".ipynb_checkpoints",
                 PROJECT/"cnn/cnn wave/.ipynb_checkpoints"]
checkpoint_sources=[]
for folder in checkpoint_dirs:
    if folder.is_dir():
        for p in folder.glob("*.ipynb"):
            if "waveform" not in p.name.lower():
                checkpoint_sources.append(p.relative_to(PROJECT).as_posix())
names+=checkpoint_sources
patterns={
    "seq20_or_final_filename":r"seq20|bilstm_dataset",
    "writer":r"\btorch\.save\b|\b(?:np|numpy)\.savez(?:_compressed)?\b",
    "36_feature_definition_or_use":r"\b36\b|feature_names|feature_columns|numeric_features|FEATURES",
    "sequence_constructor":r"def\s+\w*sequenc\w*|SEQUENCE_LENGTH|sequence_length|seq_len",
}
rx={k:re.compile(v,re.I) for k,v in patterns.items()}
source_records=[];snippets=[];literal_feature_lists=[]
for name in names:
    p=PROJECT/name
    if not p.is_file():
        source_records.append({"path":name,"exists":False});continue
    text=p.read_text(encoding="utf-8-sig")
    units=[]
    if p.suffix==".ipynb":
        nb=json.loads(text)
        # Exclude embedded plots/large outputs; inspect code and brief textual output matches only.
        for i,c in enumerate(nb["cells"]):
            if c["cell_type"]=="code":
                units.append((f"code_cell_{i}", "".join(c.get("source",[]))))
                for j,o in enumerate(c.get("outputs",[])):
                    if o.get("output_type")=="stream":
                        out="".join(o.get("text",[])) if isinstance(o.get("text"),list) else o.get("text","")
                        if rx["seq20_or_final_filename"].search(out):units.append((f"cell_{i}_stream_{j}",out))
    else:units=[("file",text)]
    counts={k:0 for k in rx}
    for unit,body in units:
        lines=body.splitlines()
        for i,line in enumerate(lines,1):
            kinds=[k for k,r in rx.items() if r.search(line)]
            for k in kinds:counts[k]+=1
            if kinds:
                snippets.append({"path":name,"unit":unit,"line":i,"matches":kinds,
                                 "text":line[:350],"context":"\n".join(lines[max(0,i-2):min(len(lines),i+1)])[:1500]})
        if p.suffix in (".py",".ipynb") and not unit.endswith("_stream"):
            try: tree=ast.parse(body)
            except SyntaxError:continue
            for node in ast.walk(tree):
                if isinstance(node,ast.Assign):
                    targets=[t.id for t in node.targets if isinstance(t,ast.Name)]
                    if any(re.search(r"feature|column|numeric",n,re.I) for n in targets):
                        try: v=ast.literal_eval(node.value)
                        except (ValueError,TypeError,SyntaxError):continue
                        if isinstance(v,(list,tuple)) and all(isinstance(x,str) for x in v):
                            literal_feature_lists.append({"path":name,"unit":unit,"line":node.lineno,
                                                         "names":targets,"length":len(v),"values":v})
    source_records.append({"path":name,"exists":True,"sha256":sha(p),"matches":counts})
# A file containing both filename and a generic writer is only a review candidate.
writer_candidates=[r["path"] for r in source_records if r.get("matches",{}).get("writer",0)
                   and r["matches"]["seq20_or_final_filename"]]
print(json.dumps({"files_read":sum(r["exists"] for r in source_records),"writer_review_candidates":writer_candidates,
                  "literal_feature_list_lengths":[{"path":r["path"],"names":r["names"],"length":r["length"]}
                                                  for r in literal_feature_lists]},indent=2))


{
  "files_read": 16,
  "writer_review_candidates": [
    "bilstm/phase2_bilstm.ipynb",
    "bilstm/phase2_bilstm_generalization_audit.ipynb"
  ],
  "literal_feature_list_lengths": [
    {
      "path": "preprocess_dl_temporal.py",
      "names": [
        "numeric_features"
      ],
      "length": 25
    },
    {
      "path": "preprocess_dl_temporal.py",
      "names": [
        "categorical_features"
      ],
      "length": 3
    },
    {
      "path": "bilstm/phase2_bilstm_generalization_audit.ipynb",
      "names": [
        "feature_records"
      ],
      "length": 0
    },
    {
      "path": "bilstm/phase2b_generalization_experiment.ipynb",
      "names": [
        "columns"
      ],
      "length": 5
    }
  ]
}


In [2]:
archive_evidence={}
for p in datasets:
    with zipfile.ZipFile(p) as z:
        entries=z.namelist()
        record={"sha256":before[p.name],"zip_entry_names":entries,
                "creation_time_utc":datetime.datetime.fromtimestamp(p.stat().st_ctime,datetime.timezone.utc).isoformat(),
                "modification_time_utc":datetime.datetime.fromtimestamp(p.stat().st_mtime,datetime.timezone.utc).isoformat()}
        if p.suffix==".pt":
            member=next(n for n in entries if n.endswith("/data.pkl") or n=="data.pkl")
            strings=[arg for op,arg,pos in pickletools.genops(z.read(member))
                     if op.name in ("UNICODE","BINUNICODE","SHORT_BINUNICODE","BINUNICODE8") and isinstance(arg,str)]
            record["pickle_string_literals"]=strings
            record["inspection"]="pickle disassembly only; tensor storage not loaded"
        else:
            record["inspection"]="ZIP member names only; schema reused from Phase 2B"
    try:
        zone=Path(str(p)+":Zone.Identifier").read_text(encoding="utf-8",errors="replace")
        fields=dict(line.split("=",1) for line in zone.splitlines() if "=" in line)
        record["download_metadata"]={"present":True,"zone_id":fields.get("ZoneId"),
            "host_url_domain":urlparse(fields.get("HostUrl","")).hostname,
            "referrer_domain":urlparse(fields.get("ReferrerUrl","")).hostname}
    except (OSError,ValueError):
        record["download_metadata"]={"present":False}
    archive_evidence[p.name]=record
# Read-only query of the project's known local IPython history, if any.
history_paths=[ROOT/".ipython/profile_default/history.sqlite",
               PROJECT/"cnn/cnn wave/.ipython/profile_default/history.sqlite"]
history_evidence=[]
for p in history_paths:
    if not p.is_file():
        history_evidence.append({"path":p.relative_to(PROJECT).as_posix(),"exists":False});continue
    try:
        with sqlite3.connect(p.as_uri()+"?mode=ro",uri=True) as con:
            rows=con.execute("SELECT session,line,source_raw FROM history WHERE lower(source_raw) LIKE ? LIMIT 40",
                             ("%bilstm_dataset_seq20%",)).fetchall()
        history_evidence.append({"path":p.relative_to(PROJECT).as_posix(),"exists":True,
                                 "exact_filename_hits":[{"session":s,"line":l,"source":t} for s,l,t in rows],
                                 "limit":40})
    except sqlite3.Error as ex:
        history_evidence.append({"path":p.relative_to(PROJECT).as_posix(),"exists":True,"error":str(ex)})
evidence={"status":"REVIEW_REQUIRED","prior_evidence":"phase2b_generalization_results/provenance_evidence.json",
          "source_records":source_records,"code_matches":snippets,"literal_feature_lists":literal_feature_lists,
          "archive_evidence":archive_evidence,"local_history":history_evidence,
          "checkpoint_directories":{p.relative_to(PROJECT).as_posix():p.is_dir() for p in checkpoint_dirs},
          "data_hashes_before":before}
(MEM/"seq20_recovery_evidence.json").write_text(json.dumps(evidence,indent=2),encoding="utf-8")
for name,r in archive_evidence.items():
    print(name, json.dumps({"entry_count":len(r["zip_entry_names"]),
                            "download_metadata":r["download_metadata"],
                            "pickle_strings":r.get("pickle_string_literals",[])}))
print("Saved bounded source evidence; no tensors deserialized or datasets changed.")


bilstm_dataset_seq20.pt {"entry_count": 18, "download_metadata": {"present": true, "zone_id": "3", "host_url_domain": null, "referrer_domain": null}, "pickle_strings": ["X_train", "storage", "0", "cpu", "y_train", "1", "X_val", "2", "y_val", "3", "X_test", "4", "y_test", "5", "y_train_raw", "6", "y_train_bin", "7", "y_val_raw", "8", "y_val_bin", "9", "y_test_raw", "10", "y_test_bin", "11"]}
bilstm_dataset_seq20.npz {"entry_count": 12, "download_metadata": {"present": true, "zone_id": "3", "host_url_domain": null, "referrer_domain": null}, "pickle_strings": []}
bilstm_dataset_seq20_station_grouped.npz {"entry_count": 12, "download_metadata": {"present": true, "zone_id": "3", "host_url_domain": null, "referrer_domain": null}, "pickle_strings": []}
Saved bounded source evidence; no tensors deserialized or datasets changed.


In [1]:
# Archive follow-up: exact path supplied by dataset Zone.Identifier metadata.
from pathlib import Path
import json,re,hashlib,subprocess,shutil,zipfile
ROOT=Path.cwd().resolve()
if (ROOT/"bilstm/bilstm_dataset").is_dir():ROOT=ROOT/"bilstm"
MEM=ROOT/".agent-memory"
evidence=json.loads((MEM/"seq20_recovery_evidence.json").read_text())
def sha(p):
    h=hashlib.sha256()
    with Path(p).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    return h.hexdigest()
refs=[]
for name in evidence["data_hashes_before"]:
    p=ROOT/"bilstm_dataset"/name
    try:raw=Path(str(p)+":Zone.Identifier").read_text(encoding="utf-8",errors="replace")
    except OSError:continue
    fields=dict(line.split("=",1) for line in raw.splitlines() if "=" in line)
    value=fields.get("ReferrerUrl","")
    if re.match(r"^[A-Za-z]:[\\/]",value):
        refs.append({"dataset":name,"referrer_path":value,"exists":Path(value).is_file()})
archive_records=[]
for archive in sorted({r["referrer_path"] for r in refs if r["exists"]}):
    executable=shutil.which("7z") or shutil.which("7zz")
    if executable:
        proc=subprocess.run([executable,"l","-slt",archive],capture_output=True,text=True,encoding="utf-8",errors="replace",check=True)
        entries=proc.stdout.split("----------",1)[-1]
        members=re.findall(r"^Path = (.+)$",entries,re.M)
        reader=lambda member:[executable,"x","-so",archive,member]
    else:
        executable=shutil.which("tar")
        if not executable:raise RuntimeError("No existing archive reader found; archive contents remain unverified.")
        proc=subprocess.run([executable,"-tf",archive],capture_output=True,text=True,encoding="utf-8",errors="replace",check=True)
        members=proc.stdout.splitlines()
        reader=lambda member:[executable,"-xOf",archive,member]
    matches=[]
    text_candidates=[]
    for member in members:
        basename=member.replace("\\","/").rstrip("/").split("/")[-1]
        if basename in evidence["data_hashes_before"]:
            h=hashlib.sha256()
            with subprocess.Popen(reader(member),stdout=subprocess.PIPE,stderr=subprocess.PIPE) as child:
                for block in iter(lambda:child.stdout.read(1024*1024),b""):h.update(block)
                error=child.stderr.read().decode("utf-8",errors="replace")
                returncode=child.wait()
            if returncode:raise RuntimeError(error[-1000:])
            matches.append({"member":member,"sha256":h.hexdigest(),
                            "matches_current_dataset":h.hexdigest()==evidence["data_hashes_before"][basename]})
        elif Path(basename).suffix.lower() in (".py",".ipynb",".json",".yaml",".yml",".md",".txt",".log"):
            text_candidates.append(member)
    archive_records.append({"path":archive,"sha256":sha(archive),"members":members,
                            "dataset_matches":matches,"source_or_metadata_candidates":text_candidates,
                            "reader":executable,"method":"Read members to stdout only; no filesystem extraction."})
evidence["download_local_references"]=refs
evidence["original_archive_followup"]=archive_records
evidence["data_hashes_after"]={name:sha(ROOT/"bilstm_dataset"/name) for name in evidence["data_hashes_before"]}
assert evidence["data_hashes_before"]==evidence["data_hashes_after"]
(MEM/"seq20_recovery_evidence.json").write_text(json.dumps(evidence,indent=2),encoding="utf-8")
print(json.dumps({"download_references":refs,"archive_records":archive_records,
                  "datasets_unchanged":True},indent=2))


{
  "download_references": [
    {
      "dataset": "bilstm_dataset_seq20.pt",
      "referrer_path": "C:\\Users\\choud\\Desktop\\bilstm_dataset.7z",
      "exists": true
    },
    {
      "dataset": "bilstm_dataset_seq20.npz",
      "referrer_path": "C:\\Users\\choud\\Desktop\\bilstm_dataset.7z",
      "exists": true
    },
    {
      "dataset": "bilstm_dataset_seq20_station_grouped.npz",
      "referrer_path": "C:\\Users\\choud\\Desktop\\bilstm_dataset.7z",
      "exists": true
    }
  ],
  "archive_records": [
    {
      "path": "C:\\Users\\choud\\Desktop\\bilstm_dataset.7z",
      "sha256": "55cfb1389c27d8956e1f8375a908617939b530367de62637dd3c4d61ce7f50e0",
      "members": [
        "bilstm_dataset\\bilstm_dataset_seq20.npz",
        "bilstm_dataset\\bilstm_dataset_seq20.pt",
        "bilstm_dataset\\bilstm_dataset_seq20_station_grouped.npz",
        "bilstm_dataset"
      ],
      "dataset_matches": [
        {
          "member": "bilstm_dataset\\bilstm_dataset_seq20.npz",
  

In [1]:
# Final review uses cached search evidence, without repeating the search.
from pathlib import Path
import json,hashlib
ROOT=Path.cwd().resolve()
if (ROOT/"bilstm/bilstm_dataset").is_dir():ROOT=ROOT/"bilstm"
PROJECT=ROOT.parent
MEM=ROOT/".agent-memory"
e=json.loads((MEM/"seq20_recovery_evidence.json").read_text())
def digest_file(p):
    h=hashlib.sha256()
    with Path(p).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    return h.hexdigest()
assert len(e["original_archive_followup"])==1
archive=e["original_archive_followup"][0]
assert len(archive["dataset_matches"])==3
assert all(m["matches_current_dataset"] for m in archive["dataset_matches"])
assert not archive["source_or_metadata_candidates"]
assert not any(h.get("exact_filename_hits") for h in e["local_history"]), "Review history matches before concluding."
assert not any(e["checkpoint_directories"].values())
assert all(digest_file(PROJECT/r["path"])==r["sha256"] for r in e["source_records"] if r["exists"])
assert digest_file(Path(archive["path"]))==archive["sha256"]
assert all(digest_file(ROOT/"bilstm_dataset"/n)==h for n,h in e["data_hashes_before"].items())
try:
    zone=Path(archive["path"]+":Zone.Identifier").read_text(encoding="utf-8",errors="replace")
    archive_download={"present":True,"field_names":[l.split("=",1)[0] for l in zone.splitlines() if "=" in l]}
except FileNotFoundError:
    archive_download={"present":False}
except OSError as exc:
    archive_download={"present":None,"read_error":type(exc).__name__}
assert archive_download["present"] is False, "Review remaining archive-origin metadata."
e["archive_download_metadata"]=archive_download
e["status"]="PARTIAL"
e["constructor_provenance"]="UNKNOWN"
e["reviewed_writer_candidates"]=[
    {"path":"bilstm/phase2_bilstm.ipynb","role":"Loads seq20; writes checkpoints and prediction cache, not the original dataset."},
    {"path":"bilstm/phase2_bilstm_generalization_audit.ipynb","role":"Loads seq20; writes Ridge prediction cache, not the original dataset."}]
e["verification"]={"datasets_unchanged":True,"searched_sources_unchanged":True,"source_archive_unchanged":True,
    "archive_members_match_all_three_datasets":True,"dataset_deserializations":0,
    "filesystem_extractions":0,"training_runs":0,"optuna_runs":0}
(MEM/"seq20_recovery_evidence.json").write_text(json.dumps(e,indent=2),encoding="utf-8")
n=sum(r["exists"] for r in e["source_records"])
table=["| Existing dataset | Referenced archive member |","|---|---|"]
for m in archive["dataset_matches"]:
    name=m["member"].replace(chr(92),"/").split("/")[-1]
    table.append(f"| {name} | Exact SHA-256 match |")
report=f"""# Seq20 provenance recovery

**PARTIAL — the referenced packaging archive was recovered and verified; the constructor and scientific provenance remain UNKNOWN.**

This report records new evidence. Prior schema, numeric-target and overlap findings remain in the [Phase 2B assessment](phase2b_generalization_results/split_provenance_report.md).

## Verified new evidence
All three dataset files have Windows Zone.Identifier metadata pointing to [Desktop/bilstm_dataset.7z](<C:/Users/choud/Desktop/bilstm_dataset.7z>). That archive exists, and its only file members are the three dataset artifacts:

{chr(10).join(table)}

Archive SHA-256: {archive['sha256']}.
Members were streamed to a hash function; no files were extracted or changed. This verifies a packaging reference, not the original author or producing program.

The archive contains **no notebook, script, README, schema, scaler, manifest or generation log**. It has no Zone.Identifier stream to identify an earlier download location. PT metadata was inspected by ZIP listing and pickle disassembly without unpickling or loading tensors; serialized strings identify array/storage keys, not feature names, source paths or a constructor.

A bounded search read **{n} named files**: relevant catalog/BiLSTM notebooks, the hybrid sequence script, root preprocessing sources/log, dataset indexes/registry and existing manifests. It covered seq20/final filenames, 36-feature references, sequence constructors, torch.save and np.savez. The only files combining seq20 references with writer calls are the two downstream Phase 2 notebooks: they load existing data and save model/prediction artifacts. Those calls do **not** construct seq20. No verified original constructor or ordered 36-feature definition was recovered.

No exact-filename generation cells were found in the two explicitly checked project-local IPython history locations; availability is recorded in the evidence. The three checked notebook-checkpoint directories are absent. No whole-project or waveform-generation scan was performed.

## Verified versus unverified
**Verified:** the referenced archive contains exact copies of all three current datasets.
**Unverified:** any connection between the legacy 15×44 preprocessing pipeline and these 20×36 tensors. A 36-channel model, loading call, file timestamp or matching sample count does not establish construction history.

Still **UNKNOWN**: exact 36-feature order/derivations; meanings and ordering/grouping of the 20 steps; physical target alignment/horizon; input/target event and station IDs/timestamps; split/cutoff/purge rules; and scaler/imputer/encoder fit rows and parameters. Existing numeric target-encoding checks do not resolve those semantics.

## Minimum missing artifacts
- Actual producing notebook/script or exported execution history, tied to the source-catalog version/hash and these output hashes.
- Ordered feature specification and window-to-input/target identity/time manifest, including target alignment and split assignments.
- Saved preprocessing parameters and fit-record IDs, or reproducible code/logs establishing where each transformation was fitted.

Recover these from the original dataset producer before reconstructing any split. No row-order mapping is assumed.

[Executed investigation notebook](seq20_provenance_investigation.ipynb) · [bounded search evidence](.agent-memory/seq20_recovery_evidence.json).
All three datasets, the referenced archive and searched sources passed final hash checks. No training, Optuna, regeneration or dataset modification occurred.
"""
(ROOT/"seq20_provenance_report.md").write_text(report,encoding="utf-8")
print(json.dumps({"status":"PARTIAL","constructor":"UNKNOWN","referenced_archive":archive["path"],
                  "all_three_dataset_hashes_match":True,"original_files_unchanged":True,
                  "report":str(ROOT/"seq20_provenance_report.md")},indent=2))


{
  "status": "PARTIAL",
  "constructor": "UNKNOWN",
  "referenced_archive": "C:\\Users\\choud\\Desktop\\bilstm_dataset.7z",
  "all_three_dataset_hashes_match": true,
  "original_files_unchanged": true,
  "report": "C:\\Users\\choud\\Desktop\\Minor project\\bilstm\\seq20_provenance_report.md"
}
